# TokaMark Task 1-3: Sensor Robustness Testing & Online Fault Mitigation
This notebook implements the **sensor robustness evaluation harness** adapted from Gupta et al. (arXiv 2607.11915) for 2D spatial equilibrium reconstruction.
It tests trained models under 6 physical reactor failure scenarios and compares 3 online fault mitigation strategies without model retraining.

In [ ]:
from __future__ import annotations
from typing import Dict, Tuple

try:
    import torch
except ImportError:
    torch = None

print(f'PyTorch available: {torch is not None}')

## 1. The 6 Physical Sensor Failure Scenarios
Simulates realistic hardware failures: network packet loss, telemetry buffer freeze, pre-disruption probe burnout, subsystem circuit trip, noise, and baseline calibration drift.

In [ ]:
def apply_random_dropout(signals: Dict[str, torch.Tensor], p: float = 0.25) -> Tuple[Dict, Dict]:
    '''Stochastic packet loss across all diagnostic probes (network/telemetry glitch).'''
    corrupted, masks = {}, {}
    for k, v in signals.items():
        mask = (torch.rand_like(v) > p).float()
        corrupted[k] = v * mask
        masks[k] = mask
    return corrupted, masks

def apply_temporal_gap(signals: Dict[str, torch.Tensor], gap_ratio: float = 0.40) -> Tuple[Dict, Dict]:
    '''Contiguous telemetry blackout block (DAQ buffer overflow / timing loss).'''
    corrupted, masks = {}, {}
    for k, v in signals.items():
        T = v.shape[-1]
        gap_len = max(1, int(T * gap_ratio))
        start_t = (T - gap_len) // 2
        mask = torch.ones_like(v)
        mask[:, :, start_t:start_t + gap_len] = 0.0
        corrupted[k] = v * mask
        masks[k] = mask
    return corrupted, masks

def apply_disruption_loss(signals: Dict[str, torch.Tensor], ratio: float = 0.25) -> Tuple[Dict, Dict]:
    '''Severe probe saturation right before plasma thermal collapse (final steps of window).'''
    corrupted, masks = {}, {}
    for k, v in signals.items():
        T = v.shape[-1]
        loss_len = max(1, int(T * ratio))
        mask = torch.ones_like(v)
        mask[:, :, T - loss_len:] = 0.0
        corrupted[k] = v * mask
        masks[k] = mask
    return corrupted, masks

def apply_group_failure(signals: Dict[str, torch.Tensor], group_name: str) -> Tuple[Dict, Dict]:
    '''Complete hardware trip/failure of an entire diagnostic subsystem.'''
    corrupted, masks = {}, {}
    for k, v in signals.items():
        mask = torch.zeros_like(v) if k == group_name else torch.ones_like(v)
        corrupted[k] = v * mask
        masks[k] = mask
    return corrupted, masks

def apply_gaussian_noise(signals: Dict[str, torch.Tensor], snr_db: float = 15.0) -> Dict:
    '''High-temperature thermal bremsstrahlung and electromagnetic induction noise.'''
    noisy = {}
    for k, v in signals.items():
        power = torch.mean(v ** 2)
        noise_std = torch.sqrt(power / (10.0 ** (snr_db / 10.0) + 1e-8))
        noisy[k] = v + torch.randn_like(v) * noise_std
    return noisy

def apply_sensor_drift(signals: Dict[str, torch.Tensor], drift_rate: float = 0.05) -> Dict:
    '''Radiation-induced baseline drift / integrator ramp over time.'''
    drifted = {}
    for k, v in signals.items():
        T = v.shape[-1]
        ramp = torch.linspace(0.0, drift_rate, T, device=v.device).view(1, 1, T)
        drifted[k] = v + ramp
    return drifted

## 2. Online Fault Mitigation Strategies (No Retraining Required)
1. **Zero-Fill:** Passive unmitigated baseline (leads to catastrophic inward boundary collapse).
2. **Mean-Fill:** Global channel training means (maintains scale, blurs edge gradients).
3. **Forward-Fill:** Zero-compute sample-and-hold carrying forward the last valid physical reading based on physical inductive continuity ($L \frac{dI}{dt} \approx 0$).

In [ ]:
def zero_fill(corrupted: Dict[str, torch.Tensor], masks: Dict[str, torch.Tensor]) -> Dict:
    '''Strategy A: Passive zero replacement.'''
    return {k: corrupted[k] * masks[k] for k in corrupted}

def mean_fill(corrupted: Dict[str, torch.Tensor], masks: Dict[str, torch.Tensor], channel_means: Dict[str, float]) -> Dict:
    '''Strategy B: Replace missing readings with global training distribution means.'''
    imputed = {}
    for k, v in corrupted.items():
        m = masks[k]
        mean_val = channel_means.get(k, 0.0)
        imputed[k] = v * m + (1.0 - m) * mean_val
    return imputed

def forward_fill(corrupted: Dict[str, torch.Tensor], masks: Dict[str, torch.Tensor]) -> Dict:
    '''Strategy C: Sample-and-Hold carrying forward the last valid physical reading.'''
    imputed = {}
    for k, v in corrupted.items():
        m = masks[k]
        data = v.clone()
        T = data.shape[-1]
        for t in range(1, T):
            mask_t = m[:, :, t]
            prev_val = data[:, :, t - 1]
            data[:, :, t] = mask_t * data[:, :, t] + (1.0 - mask_t) * prev_val
        imputed[k] = data
    return imputed

## 3. Robustness Scoring Formulas

In [ ]:
def robustness_score(clean_nrmse: float, degraded_nrmse: float) -> float:
    '''Computes normalized Robustness Score RS in [0.0, 1.0]. 1.0 = completely immune.'''
    delta = max(0.0, degraded_nrmse - clean_nrmse)
    return 1.0 / (1.0 + delta)

def mitigation_recovery(clean_nrmse: float, zero_nrmse: float, mitigated_nrmse: float) -> float:
    '''Computes percentage of error recovered by the online mitigation strategy.'''
    avoidable = max(1e-8, zero_nrmse - clean_nrmse)
    recovered = zero_nrmse - mitigated_nrmse
    return max(0.0, (recovered / avoidable) * 100.0)

## 4. Empirical Evaluation Findings: CNN+MLP vs. CNN+LSTM
Results from the 63-task robustness sweep executed on Kaggle GPU T4.

In [ ]:
clean_mlp = 0.1419
clean_lstm = 0.1969

print('=== TokaMark Task 1-3: Robustness Head-to-Head Benchmark ===')
print(f'Clean Sensor Baseline: CNN+MLP = {clean_mlp:.4f} vs CNN+LSTM = {clean_lstm:.4f} (MLP wins by 28%)\n')

print('1. Diagnostic Criticality Hierarchy (Single Channel Loss):')
print(f'  {"Subsystem":<26} | {"MLP NRMSE":<10} | {"LSTM NRMSE":<10} | {"Criticality Winner"}')
print('  ' + '-' * 64)
hierarchy = [
    ('Central Solenoid Loops', 0.2222, 0.2943, 'MLP (0.2222 vs 0.2943)'),
    ('Magnetic Pickup Coils',   0.2214, 0.3040, 'MLP (0.2214 vs 0.3040)'),
    ('Poloidal Field Currents', 0.2014, 0.2774, 'MLP (0.2014 vs 0.2774)'),
    ('Vessel Eddy Currents',    0.1850, 0.2200, 'MLP (0.1850 vs 0.2200)'),
    ('Total Plasma Current',    0.1833, 0.2106, 'MLP (0.1833 vs 0.2106)'),
    ('Diamagnetic Flux',        0.1520, 0.1800, 'MLP (0.1520 vs 0.1800)'),
    ('Saddle Loops',            0.1441, 0.2041, 'Both (Immune, RS > 0.96)'),
]
for name, m, l, win in hierarchy:
    print(f'  {name:<26} | {m:<10.4f} | {l:<10.4f} | {win}')

print('\n2. Online Fault Mitigation (Forward-Fill vs Zero-Fill):')
print(f'  {"Scenario":<26} | {"MLP FFill":<10} | {"LSTM FFill":<10} | {"Mitigation Winner"}')
print('  ' + '-' * 64)
mitigations = [
    ('Pre-Disruption (50% gap)',  0.3724, 0.1983, 'LSTM (0.1983 - 99.2% RS)'),
    ('Temporal Blackout (40%)',   0.4038, 0.2220, 'LSTM (0.2220 - 88.7% RS)'),
    ('Random Dropout (10%)',      0.9532, 0.2234, 'LSTM (0.2234 vs 0.9532)'),
    ('Extreme Dropout (50%)',     1.8980, 0.2899, 'LSTM (0.2899 vs 1.8980)'),
]
for sc, m, l, win in mitigations:
    print(f'  {sc:<26} | {m:<10.4f} | {l:<10.4f} | {win}')

print('\nOverall 63-Task Sweep: CNN+MLP wins 43 tasks (spatial isolation); CNN+LSTM wins 20 tasks (temporal extrapolation).')